# AIC — Notebook 06: Build Qdrant Text Index

Encodes OCR / Caption / ASR JSON files with BGE-M3 and uploads to Qdrant.

**Prerequisites:** Run notebooks 02, 03, 04 first (or any subset)
**Output:** Qdrant running at localhost:6333 with collections: `captions`, `ocr`, `asr`

**Note:** Start Qdrant server first with Docker:  
`!docker run -d -p 6333:6333 qdrant/qdrant`

In [ ]:
import subprocess, sys, os
GITHUB_REPO = "https://github.com/YOUR_USERNAME/AIC_System.git"
REPO_DIR = "/kaggle/working/AIC_System"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git","clone","--depth","1",GITHUB_REPO,REPO_DIR],check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"pull"],check=True)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",f"{REPO_DIR}/requirements.txt"],check=True)
# Start Qdrant server
subprocess.Popen(['docker','run','-d','-p','6333:6333','qdrant/qdrant'])
import time; time.sleep(5)
print('Setup complete.')

In [ ]:
from pathlib import Path
OCR_DIR      = Path('/kaggle/working/ocr')
CAPTIONS_DIR = Path('/kaggle/working/captions')
ASR_DIR      = Path('/kaggle/working/subtitles')
QDRANT_URL   = 'http://localhost:6333'
print(f'OCR files:      {len(list(OCR_DIR.glob("*.json"))) if OCR_DIR.exists() else 0}')
print(f'Caption files:  {len(list(CAPTIONS_DIR.glob("*.json"))) if CAPTIONS_DIR.exists() else 0}')
print(f'ASR files:      {len(list(ASR_DIR.glob("*.json"))) if ASR_DIR.exists() else 0}')

In [ ]:
from src.embeddings.text.bge import BGEEncoder
encoder = BGEEncoder()
encoder.load()
print('BGE-M3 ready.')

In [ ]:
from src.database.qdrant_db import QdrantDB
db = QdrantDB(url=QDRANT_URL)
db.connect()
db.create_collections(overwrite=True)
print('Qdrant collections created.')

In [ ]:
import time
t0 = time.time()
if CAPTIONS_DIR.exists() and list(CAPTIONS_DIR.glob('*.json')):
    n = db.index_from_json(str(CAPTIONS_DIR),'captions','caption_en',encoder)
    print(f'Captions: {n:,} vectors')
if OCR_DIR.exists() and list(OCR_DIR.glob('*.json')):
    n = db.index_from_json(str(OCR_DIR),'ocr','texts',encoder)
    print(f'OCR: {n:,} vectors')
if ASR_DIR.exists() and list(ASR_DIR.glob('*.json')):
    n = db.index_from_json(str(ASR_DIR),'asr','asr_text',encoder)
    print(f'ASR: {n:,} vectors')
print(f'Total time: {time.time()-t0:.0f}s')
for col in ['captions','ocr','asr']:
    try: print(f'  {col}: {db.collection_count(db.COLLECTIONS[col]):,} pts')
    except: pass